In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.ensemble import RandomForestRegressor, BaggingRegressor, AdaBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, Conv1D, MaxPooling1D, Flatten, GRU
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

2025-10-11 05:38:46.674086: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760161126.700639     181 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760161126.708714     181 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
df_combined = pd.read_csv('/kaggle/working/clean_dataset.csv')

In [3]:
X_COLS = ['open', 'high', 'low', 'volume', 'market_cap', 'url_shares',
          'unique_url_shares', 'reddit_posts', 'reddit_posts_score',
          'reddit_comments', 'reddit_comments_score', 'tweets', 'tweet_spam',
          'tweet_followers', 'tweet_quotes', 'tweet_retweets', 'tweet_replies',
          'tweet_favorites', 'tweet_sentiment1', 'tweet_sentiment2',
          'tweet_sentiment3', 'tweet_sentiment4', 'tweet_sentiment5',
          'tweet_sentiment_impact1', 'tweet_sentiment_impact2',
          'tweet_sentiment_impact3', 'tweet_sentiment_impact4',
          'tweet_sentiment_impact5', 'social_score', 'average_sentiment', 'news',
          'price_score', 'social_impact_score', 'correlation_rank',
          'galaxy_score', 'volatility', 'market_cap_rank',
          'percent_change_24h_rank', 'volume_24h_rank', 'social_volume_24h_rank',
          'social_score_24h_rank', 'medium', 'youtube', 'social_volume',
          'percent_change_24h', 'market_cap_global']
Y_COL = 'close'

In [4]:
def print_metrics(y_true, y_pred, model_name):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    
    print("-" * 50)
    print(f"Model: {model_name}")
    print(f"R-squared (R2): {r2:.4f}")
    print(f"Mean Absolute Error (MAE): {mae:.4f}")
    print(f"Mean Squared Error (MSE): {mse:.4f}")
    print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
    print("-" * 50)

In [5]:
TIME_STEPS = 10

In [6]:
try:
    X = df_combined[X_COLS].values
    y = df_combined[Y_COL].values
except NameError:
    print("FATAL ERROR: df_combined is not defined. Please ensure your DataFrame is loaded.")
    raise

In [7]:
split_point = int(len(X) * 0.8)

X_train_ml, X_test_ml = X[:split_point], X[split_point:]
y_train_ml, y_test_ml = y[:split_point], y[split_point:]

In [8]:
scaler_ml = StandardScaler()
X_train_ml_scaled = scaler_ml.fit_transform(X_train_ml)
X_test_ml_scaled = scaler_ml.transform(X_test_ml)

In [9]:
print("\n--- Starting Traditional Machine Learning Model Training ---")


--- Starting Traditional Machine Learning Model Training ---


In [10]:
try:
    lr_model = LinearRegression()
    lr_model.fit(X_train_ml_scaled, y_train_ml)
    y_pred_lr = lr_model.predict(X_test_ml_scaled)
    print_metrics(y_test_ml, y_pred_lr, "Linear Regression")
except Exception as e:
    print(f"Error training Linear Regression: {e}")

--------------------------------------------------
Model: Linear Regression
R-squared (R2): 1.0000
Mean Absolute Error (MAE): 28.7759
Mean Squared Error (MSE): 4971.5255
Root Mean Squared Error (RMSE): 70.5090
--------------------------------------------------


In [11]:
try:
    # Set max_iter for convergence
    sgd_model = SGDRegressor(max_iter=1000, tol=1e-3, random_state=42)
    sgd_model.fit(X_train_ml_scaled, y_train_ml)
    y_pred_sgd = sgd_model.predict(X_test_ml_scaled)
    print_metrics(y_test_ml, y_pred_sgd, "SGDRegressor")
except Exception as e:
    print(f"Error training SGDRegressor: {e}")

--------------------------------------------------
Model: SGDRegressor
R-squared (R2): 0.9999
Mean Absolute Error (MAE): 82.3104
Mean Squared Error (MSE): 13435.5305
Root Mean Squared Error (RMSE): 115.9117
--------------------------------------------------


In [12]:
try:
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    rf_model.fit(X_train_ml_scaled, y_train_ml)
    y_pred_rf = rf_model.predict(X_test_ml_scaled)
    print_metrics(y_test_ml, y_pred_rf, "Random Forest Regressor")
except Exception as e:
    print(f"Error training Random Forest Regressor: {e}")

--------------------------------------------------
Model: Random Forest Regressor
R-squared (R2): 0.9999
Mean Absolute Error (MAE): 44.1905
Mean Squared Error (MSE): 16010.5997
Root Mean Squared Error (RMSE): 126.5330
--------------------------------------------------


In [13]:
try:
    # Using Random Forest as the base estimator, though default is DecisionTreeRegressor
    bagging_model = BaggingRegressor(estimator=RandomForestRegressor(n_estimators=10), n_estimators=10, random_state=42, n_jobs=-1)
    bagging_model.fit(X_train_ml_scaled, y_train_ml)
    y_pred_bag = bagging_model.predict(X_test_ml_scaled)
    print_metrics(y_test_ml, y_pred_bag, "Bagging Regressor")
except Exception as e:
    print(f"Error training Bagging Regressor: {e}")

--------------------------------------------------
Model: Bagging Regressor
R-squared (R2): 0.9998
Mean Absolute Error (MAE): 48.5140
Mean Squared Error (MSE): 20753.9665
Root Mean Squared Error (RMSE): 144.0624
--------------------------------------------------


In [14]:
try:
    # Base estimator is DecisionTreeRegressor(max_depth=3) by default
    ada_model = AdaBoostRegressor(n_estimators=50, learning_rate=1.0, random_state=42)
    ada_model.fit(X_train_ml_scaled, y_train_ml)
    y_pred_ada = ada_model.predict(X_test_ml_scaled)
    print_metrics(y_test_ml, y_pred_ada, "AdaBoost Regressor")
except Exception as e:
    print(f"Error training AdaBoost Regressor: {e}")

--------------------------------------------------
Model: AdaBoost Regressor
R-squared (R2): 0.9960
Mean Absolute Error (MAE): 527.4880
Mean Squared Error (MSE): 464935.9002
Root Mean Squared Error (RMSE): 681.8621
--------------------------------------------------


In [15]:
def create_sequences(X_data, y_data, time_steps):
    """
    Creates sequences for time series prediction.
    X_data: (samples, features)
    y_data: (samples,)
    Returns: X_seq (samples, time_steps, features), y_seq (samples, 1)
    """
    X_seq, y_seq = [], []
    for i in range(len(X_data) - time_steps):
        X_seq.append(X_data[i:(i + time_steps)])
        y_seq.append(y_data[i + time_steps])
    return np.array(X_seq), np.array(y_seq).reshape(-1, 1)

In [16]:
X_dl = df_combined[X_COLS].values
y_dl = df_combined[Y_COL].values

In [17]:
scaler_X_dl = StandardScaler()
X_dl_scaled = scaler_X_dl.fit_transform(X_dl)

scaler_y_dl = StandardScaler()
y_dl_scaled = scaler_y_dl.fit_transform(y_dl.reshape(-1, 1))

In [18]:
X_seq, y_seq = create_sequences(X_dl_scaled, y_dl_scaled, TIME_STEPS)

In [19]:
split_idx = int(len(X_seq) * 0.8)
X_train_dl, X_test_dl = X_seq[:split_idx], X_seq[split_idx:]
y_train_dl, y_test_dl = y_seq[:split_idx], y_seq[split_idx:]

N_FEATURES = X_train_dl.shape[2]
EPOCHS = 50 # Keep epochs low for fast execution, increase for actual training
BATCH_SIZE = 32
tf.random.set_seed(42) # Set seed for reproducibility

In [20]:
es_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [21]:
print("\n--- Starting Deep Learning Model Training (Time Series) ---")


--- Starting Deep Learning Model Training (Time Series) ---


In [22]:
def train_and_evaluate_dl(model, X_train, y_train, X_test, y_test, scaler_y, model_name):
    print(f"\nTraining {model_name}...")
    try:
        model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
        model.fit(
            X_train, y_train,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            validation_split=0.1,
            callbacks=[es_callback],
            verbose=0
        )
        
        # Predict and inverse transform
        y_pred_scaled = model.predict(X_test, verbose=0)
        y_test_unscaled = scaler_y.inverse_transform(y_test)
        y_pred_unscaled = scaler_y.inverse_transform(y_pred_scaled)
        
        print_metrics(y_test_unscaled, y_pred_unscaled, model_name)
        return model
    except Exception as e:
        print(f"Error training {model_name}: {e}")
        return None

In [23]:
lstm_model = Sequential([
    LSTM(50, activation='relu', input_shape=(TIME_STEPS, N_FEATURES)),
    Dropout(0.2),
    Dense(1)
])
lstm_model.summary()
train_and_evaluate_dl(lstm_model, X_train_dl, y_train_dl, X_test_dl, y_test_dl, scaler_y_dl, "LSTM Regression")

2025-10-11 05:43:11.632292: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 50)             │        19,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 50)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,451 (75.98 KB)

 Trainable params: 19,451 (75.98 KB)

 Non-trainable params: 0 (0.00 B)


Training LSTM Regression...
--------------------------------------------------
Model: LSTM Regression
R-squared (R2): -0.0318
Mean Absolute Error (MAE): 6904.4742
Mean Squared Error (MSE): 121836801.2517
Root Mean Squared Error (RMSE): 11037.9709
--------------------------------------------------


<Sequential name=sequential, built=True>

In [24]:
bilstm_model = Sequential([
    Bidirectional(LSTM(50, activation='relu'), input_shape=(TIME_STEPS, N_FEATURES)),
    Dropout(0.2),
    Dense(1)
])
bilstm_model.summary()
train_and_evaluate_dl(bilstm_model, X_train_dl, y_train_dl, X_test_dl, y_test_dl, scaler_y_dl, "Bi-LSTM Regression")

/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 100)            │        38,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           101 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 38,901 (151.96 KB)

 Trainable params: 38,901 (151.96 KB)

 Non-trainable params: 0 (0.00 B)


Training Bi-LSTM Regression...
--------------------------------------------------
Model: Bi-LSTM Regression
R-squared (R2): -0.0382
Mean Absolute Error (MAE): 6980.0331
Mean Squared Error (MSE): 122597302.4575
Root Mean Squared Error (RMSE): 11072.3666
--------------------------------------------------


<Sequential name=sequential_1, built=True>

In [25]:
cnn_model = Sequential([
    Conv1D(filters=64, kernel_size=2, activation='relu', input_shape=(TIME_STEPS, N_FEATURES)),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(50, activation='relu'),
    Dropout(0.2),
    Dense(1)
])
cnn_model.summary()
train_and_evaluate_dl(cnn_model, X_train_dl, y_train_dl, X_test_dl, y_test_dl, scaler_y_dl, "CNN Regression (1D)")

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 9, 64)          │         5,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 4, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 50)             │        12,850 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 50)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 18,853 (73.64 KB)

 Trainable params: 18,853 (73.64 KB)

 Non-trainable params: 0 (0.00 B)


Training CNN Regression (1D)...
--------------------------------------------------
Model: CNN Regression (1D)
R-squared (R2): -0.2249
Mean Absolute Error (MAE): 7580.7610
Mean Squared Error (MSE): 144639377.3789
Root Mean Squared Error (RMSE): 12026.6112
--------------------------------------------------


<Sequential name=sequential_2, built=True>

In [26]:
input_layer = tf.keras.Input(shape=(TIME_STEPS, N_FEATURES))
x = LSTM(50, activation='relu', return_sequences=True)(input_layer)
x = GRU(50, activation='relu')(x)
x = Dropout(0.2)(x)
output_layer = Dense(1)(x)

lstm_gru_model = Model(inputs=input_layer, outputs=output_layer, name='LSTM_GRU_Hybrid')
lstm_gru_model.summary()
train_and_evaluate_dl(lstm_gru_model, X_train_dl, y_train_dl, X_test_dl, y_test_dl, scaler_y_dl, "Ensemble Hybrid (LSTM+GRU)")

Model: "LSTM_GRU_Hybrid"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 10, 46)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 10, 50)         │        19,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 50)             │        15,300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 50)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 34,751 (135.75 KB)

 Trainable params: 34,751 (135.75 KB)

 Non-trainable params: 0 (0.00 B)


Training Ensemble Hybrid (LSTM+GRU)...
--------------------------------------------------
Model: Ensemble Hybrid (LSTM+GRU)
R-squared (R2): -0.0060
Mean Absolute Error (MAE): 6746.6496
Mean Squared Error (MSE): 118787434.2445
Root Mean Squared Error (RMSE): 10898.9648
--------------------------------------------------


<Functional name=LSTM_GRU_Hybrid, built=True>

In [27]:
input_layer = tf.keras.Input(shape=(TIME_STEPS, N_FEATURES))
x = Conv1D(filters=64, kernel_size=2, activation='relu')(input_layer)
x = MaxPooling1D(pool_size=2)(x)
x = Bidirectional(LSTM(50, activation='relu'))(x)
x = Dropout(0.2)(x)
output_layer = Dense(1)(x)

bilstm_cnn_model = Model(inputs=input_layer, outputs=output_layer, name='BiLSTM_CNN_Hybrid')
bilstm_cnn_model.summary()
train_and_evaluate_dl(bilstm_cnn_model, X_train_dl, y_train_dl, X_test_dl, y_test_dl, scaler_y_dl, "Ensemble Hybrid (Bi-LSTM+CNN)")

print("\n--- Model Training Complete ---")

Model: "BiLSTM_CNN_Hybrid"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 10, 46)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 9, 64)          │         5,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 4, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 100)            │        46,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           101 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 52,053 (203.33 KB)

 Trainable params: 52,053 (203.33 KB)

 Non-trainable params: 0 (0.00 B)


Training Ensemble Hybrid (Bi-LSTM+CNN)...
--------------------------------------------------
Model: Ensemble Hybrid (Bi-LSTM+CNN)
R-squared (R2): -0.0204
Mean Absolute Error (MAE): 6635.8378
Mean Squared Error (MSE): 120494550.5677
Root Mean Squared Error (RMSE): 10977.0010
--------------------------------------------------

--- Model Training Complete ---
